# Исправление get_atm()

## Проверка как можно избежать транспонирования матрицы в функции get_atm

In [1]:
import tracemalloc

import numpy as np
import pandas as pd

In [2]:
%%time

# Начать отслеживание
tracemalloc.start()

# Код:
d = {}

s1 = pd.Series({1:'a', 2:'b'})
s2 = pd.Series({3:'c', 4:'d'})
s3 = pd.Series({1:'a1', 3:'c1'})
d['q'] = s1
d['w'] = s2
d['e'] = s3

df = pd.DataFrame(d).T

# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

# Печать результата - датафрейма
df

Текущее использование: 0.03 MB
Пиковое использование: 0.04 MB
CPU times: total: 0 ns
Wall time: 12 ms


,1,2,3,4
q,a,b,NaN,NaN
w,NaN,NaN,c,d
e,a1,NaN,c1,NaN


#### Вариант 1

In [3]:
%%time

# Начать отслеживание
tracemalloc.start()

# Код:
df2 = pd.DataFrame()

dft = pd.DataFrame()
dft['q'] = s1
df2 = pd.concat([df2, dft.T])

dft = pd.DataFrame()
dft['w'] = s2
df2 = pd.concat([df2, dft.T])

dft = pd.DataFrame()
dft['e'] = s3
df2 = pd.concat([df2, dft.T])

# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

# Печать результата - датафрейма
df2

Текущее использование: 0.04 MB
Пиковое использование: 0.06 MB
CPU times: total: 31.2 ms
Wall time: 32 ms


,1,2,3,4
q,a,b,NaN,NaN
w,NaN,NaN,c,d
e,a1,NaN,c1,NaN


#### Вариант 2 

    pd.DataFrame.from_dict(d, orient='index')

In [4]:
%%time

# Начать отслеживание
tracemalloc.start()

# Код:
d = {}

d['q'] = s1
d['w'] = s2
d['e'] = s3

# Вместо транспонирования:
df3 = pd.DataFrame.from_dict(d, orient='index')

# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

# Печать результата - датафрейма
df3


Текущее использование: 0.01 MB
Пиковое использование: 0.01 MB
CPU times: total: 0 ns
Wall time: 5 ms


,1,2,3,4
q,a,b,NaN,NaN
e,a1,NaN,c1,NaN
w,NaN,NaN,c,d


Пожалуй это идеальное решение!

#### Вариант 3

Последовательное добавление результатов расчета в матрицу, без создания общего словаря или списка

In [5]:
%%time

# Начать отслеживание
tracemalloc.start()

# Код:
df4 = pd.DataFrame()

dft = pd.DataFrame.from_dict({'q': {1:'a', 2:'b'}}, orient='index')
df4 = pd.concat([df4, dft])
dft = pd.DataFrame.from_dict({'w': {3:'c', 4:'d'}}, orient='index')
df4 = pd.concat([df4, dft])
dft = pd.DataFrame.from_dict({'e': {1:'a1', 3:'c1', 5:'e'}}, orient='index')
df4 = pd.concat([df4, dft])


# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

# Печать результата - датафрейма
df4

Текущее использование: 0.02 MB
Пиковое использование: 0.04 MB
CPU times: total: 31.2 ms
Wall time: 27 ms


,1,2,3,4,5
q,a,b,NaN,NaN,NaN
w,NaN,NaN,c,d,NaN
e,a1,NaN,c1,NaN,e


Достоинство данного подхода в том, что не забивается память. На каждой итерации данные добавляются сразу в матрицу, а не накапливаются в словаре или списке.

# Тесты расчетов с использованием разных подходов.

Задачи:
1. Выяснить возможно ли посчитать крупную матрицу, без использщования транспонирования матрицы
2. Какой из подходов расходует меньше ресурсов и потенциально позволяет обсчитывать большее количество объектов

## Загрузка библиотек

In [6]:
import osmnx as ox
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import genesis as genesis


# Версии библиотек
print('OSMnx:', ox.__version__,
      '\nGeoPandas:', gpd.__version__,
      '\nPandas:', pd.__version__,
      '\ngenesis:', genesis.__version__,
      )

OSMnx: 2.0.1 
GeoPandas: 1.0.1 
Pandas: 2.1.1 
genesis: 0.2.200


## Загрузка данных

In [7]:
from genesis.states import FirstArrivalUnitState
from graphs.speeds import set_graph_travel_times, kmh_to_mm
from fire_units.settings import DEFAULT_SPEEDS


# 1. Загрузка исходных данных
## Загрузка ГДС
G = ox.load_graphml('C:/Users/Obsidian/Desktop/Красноярск Демо/______________ef3f7396_30d9_4d94_b617_55ad61b06e30.ml')

## Загрузка зданий
buildings = gpd.read_file('C:/Users/Obsidian/Desktop/Красноярск Демо/data/bld.gpkg')
### Сопоставление зданий с ближайшими узлами ГДС
centroids = ox.projection.project_gdf(buildings).geometry.centroid
buildings['node'] = ox.distance.nearest_nodes(ox.project_graph(G), 
                                              centroids.x,
                                              centroids.y
                                              )

## Варианты

### Вариант 2 

    pd.DataFrame.from_dict(d, orient='index')

#### Модифицированная функция get_atm()

In [8]:
import networkx as nx

from genesis.swiss_knife import DELAY_TIME
from genesis.utils import Progressbar


def get_atm(G,
            data: gpd.GeoDataFrame  = None,
            data_sample_size: int   = None,
            data_node_field: str    = 'node',
            weight: str             = 'travel_time',
            cutoff: float           = None,
            data_cutoff_field: str  = None,
            delay: float            = DELAY_TIME,
            target_set: set         = None,
            print_calc_states: bool = True,
            calc_function: callable = None,
            ):
    '''
    Расчет матрицы времен прибытия.

    Аргументы:

        `G`: nx.MultiDiGraph
            Граф улично-дорожной сети

        `data`: pd.DataFrame = None
            Данные для расчета. Может быть указан набор данных для которых следует рассчитать,
            например, здания, или набор узлов графа.
            По умолчанию используются все узлы графа

        `data_sample_size`: int = None
            Размер выборки данных для расчета. Количество записей
            data ,которые будут учтены при расчете.
        
        `data_node_field`: str   = 'node'
            Поле в котором хранится индекс узла, для которого производится расчет.

        `weight`: str или callable  = "travel_time"
            Имя поля содержащего вес ребер, или функция позволяющая вычислять 
            вес динамически.

        `cutoff`: float = None,
            Размер расчетной области. По умолчанию производится расчет для всего графа.

        `delay`: float, optional = None
            Задержка в расчете. Например на обслуживание вызова на пожар.
            По умолчанию указана в swiss_knife.DELAY_TIME

        `target_set`: set = None
            Целевой сет узлов графа, которые рассматриваются в качестве потенциальных мест размещения.

        `print_calc_states`: bool = True
            Флаг отображения прогресса расчета.

    Возвращает:

        `matrix`: pd.DataFrame
            Время прибытия первого подразделения в каждый из узлов графа. 
            Столбцами являются узлы графа, индексами - узлы графа, до которых производился расчет.
    '''

    # Если конкретный набор данных не передан, используем узлы графа
    if data is None:
        data = ox.graph_to_gdfs(G, edges=False)
        data[data_node_field] = data.index

    # Если указана доля набора данных, выбираем его случайным образом из data
    if not data_sample_size is None:
        data = data.sample(data_sample_size)

    # Реверсный граф
    GR = nx.reverse(G, copy=True)

    # перебираем все записи в наборе данных
    if print_calc_states:
        pb = Progressbar(len(data), bins=40)
    d = {}
    i = 0
    for di, dt in data.iterrows():
        node = dt[data_node_field]
        if not data_cutoff_field is None:
            ctf = dt[data_cutoff_field]
        else:
            ctf = cutoff - delay if not cutoff is None else None
        length = nx.single_source_dijkstra_path_length(
            GR,
            source = node,
            cutoff = ctf,
            weight = weight,
            )
        # Оставляем только узлы в которых можно разместить [предполагалось удалить позже, но почему, пока не понятно]
        if not target_set is None:
            length = {k:v for k,v in length.items() if k in target_set}
            
        d[di] = pd.Series(length) + delay
        # d[node] = pd.Series(length) + delay
        if print_calc_states:
            pb()
        if calc_function:
            value = i / len(data)
            calc_function(value = value)

        i += 1

    # Формирование матрицы
    # matrix = pd.DataFrame(d).T
    matrix = pd.DataFrame.from_dict(d, orient='index')

    # Удаление зданий, к которым невозможно прибытия из перечня приемлемых узлов
    if not target_set is None:
        # Оставляем только узлы в которых можно разместить
        # matrix = matrix[list(set(matrix.columns) & target_set)]
        # Здания к которым невозможно прибыть после отброса
        zero_buildings = np.sum(matrix, axis=1)==0
        zero_buildings_count = sum(zero_buildings)
        if zero_buildings_count > 0:
            print(f'!{zero_buildings_count} зданий не доступны из указанных мест размещения!')
            matrix = matrix[zero_buildings == False]

    del GR
    return matrix

#### Тест расчета

In [9]:
print(f'Зданий в расчете:  {len(buildings)}')
print(f'Узлов:             {G.number_of_nodes()}')
print(f'Ребер:             {G.number_of_edges()}')

Зданий в расчете:  58094
Узлов:             99532
Ребер:             430595


In [10]:
%%time

# Начать отслеживание
tracemalloc.start()

# Тестируемый код
matrix = get_atm(
    G,
    data             = buildings,
    cutoff           = 10,
    data_sample_size = 10000
)

# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

|########################################| 100.0%
Текущее использование: 6929.35 MB
Пиковое использование: 19487.02 MB
CPU times: total: 1h 35min 17s
Wall time: 1h 36min 9s


In [11]:
matrix.shape

(10000, 90799)

### Вариант 3

Последовательное добавление результатов расчета в матрицу, без создания общего словаря или списка

#### Модифицированная функция get_atm()

In [ ]:
import networkx as nx

from genesis.swiss_knife import DELAY_TIME
from genesis.utils import Progressbar


def get_atm(G,
            data: gpd.GeoDataFrame  = None,
            data_sample_size: int   = None,
            data_node_field: str    = 'node',
            weight: str             = 'travel_time',
            cutoff: float           = None,
            data_cutoff_field: str  = None,
            delay: float            = DELAY_TIME,
            target_set: set         = None,
            print_calc_states: bool = True,
            calc_function: callable = None,
            ):
    '''
    Расчет матрицы времен прибытия.

    Аргументы:

        `G`: nx.MultiDiGraph
            Граф улично-дорожной сети

        `data`: pd.DataFrame = None
            Данные для расчета. Может быть указан набор данных для которых следует рассчитать,
            например, здания, или набор узлов графа.
            По умолчанию используются все узлы графа

        `data_sample_size`: int = None
            Размер выборки данных для расчета. Количество записей
            data ,которые будут учтены при расчете.
        
        `data_node_field`: str   = 'node'
            Поле в котором хранится индекс узла, для которого производится расчет.

        `weight`: str или callable  = "travel_time"
            Имя поля содержащего вес ребер, или функция позволяющая вычислять 
            вес динамически.

        `cutoff`: float = None,
            Размер расчетной области. По умолчанию производится расчет для всего графа.

        `delay`: float, optional = None
            Задержка в расчете. Например на обслуживание вызова на пожар.
            По умолчанию указана в swiss_knife.DELAY_TIME

        `target_set`: set = None
            Целевой сет узлов графа, которые рассматриваются в качестве потенциальных мест размещения.

        `print_calc_states`: bool = True
            Флаг отображения прогресса расчета.

    Возвращает:

        `matrix`: pd.DataFrame
            Время прибытия первого подразделения в каждый из узлов графа. 
            Столбцами являются узлы графа, индексами - узлы графа, до которых производился расчет.
    '''

    # Если конкретный набор данных не передан, используем узлы графа
    if data is None:
        data = ox.graph_to_gdfs(G, edges=False)
        data[data_node_field] = data.index

    # Если указана доля набора данных, выбираем его случайным образом из data
    if not data_sample_size is None:
        data = data.sample(data_sample_size)

    # Реверсный граф
    GR = nx.reverse(G, copy=False)

    # Матрица
    matrix = pd.DataFrame()

    # перебираем все записи в наборе данных
    if print_calc_states:
        pb = Progressbar(len(data), bins=40)
    # d = {}
    i = 0
    for di, dt in data.iterrows():
        node = dt[data_node_field]
        if not data_cutoff_field is None:
            ctf = dt[data_cutoff_field]
        else:
            ctf = cutoff - delay if not cutoff is None else None
        length = nx.single_source_dijkstra_path_length(
            GR,
            source = node,
            cutoff = ctf,
            weight = weight,
            )
        # Оставляем только узлы в которых можно разместить [предполагалось удалить позже, но почему, пока не понятно]
        if not target_set is None:
            length = {k:v for k,v in length.items() if k in target_set}

        length = {k:v+delay for k,v in length.items()}
        dft = pd.DataFrame.from_dict({di: length}, orient='index')
        matrix = pd.concat([matrix, dft])
        # d[di] = pd.Series(length) + delay
        # d[node] = pd.Series(length) + delay
        if print_calc_states:
            pb()
        if calc_function:
            value = i / len(data)
            calc_function(value = value)

        i += 1

    # Формирование матрицы
    # matrix = pd.DataFrame(d).T
    # matrix = pd.DataFrame.from_dict(d, orient='index')

    # Удаление зданий, к которым невозможно прибытия из перечня приемлемых узлов
    if not target_set is None:
        # Оставляем только узлы в которых можно разместить
        # matrix = matrix[list(set(matrix.columns) & target_set)]
        # Здания к которым невозможно прибыть после отброса
        zero_buildings = np.sum(matrix, axis=1)==0
        zero_buildings_count = sum(zero_buildings)
        if zero_buildings_count > 0:
            print(f'!{zero_buildings_count} зданий не доступны из указанных мест размещения!')
            matrix = matrix[zero_buildings == False]

    G = nx.reverse(GR, copy=False)
    return matrix

#### Тест расчета

In [ ]:
%%time

# Начать отслеживание
tracemalloc.start()

# Тестируемый код
matrix = get_atm(
    G,
    data             = buildings,
    cutoff           = 10,
    data_sample_size = 10000
)

# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

In [ ]:
matrix.shape

### Вариант 3+(батчи)

Последовательное добавление результатов расчета в матрицу, без создания общего словаря или списка

#### Модифицированная функция get_atm()

In [ ]:
import networkx as nx

from genesis.swiss_knife import DELAY_TIME
from genesis.utils import Progressbar


def get_atm(G,
            data: gpd.GeoDataFrame  = None,
            data_sample_size: int   = None,
            data_node_field: str    = 'node',
            weight: str             = 'travel_time',
            cutoff: float           = None,
            data_cutoff_field: str  = None,
            delay: float            = DELAY_TIME,
            target_set: set         = None,
            print_calc_states: bool = True,
            calc_function: callable = None,
            batch_size: int         = 1000,
            ):
    '''
    Расчет матрицы времен прибытия.

    Аргументы:

        `G`: nx.MultiDiGraph
            Граф улично-дорожной сети

        `data`: pd.DataFrame = None
            Данные для расчета. Может быть указан набор данных для которых следует рассчитать,
            например, здания, или набор узлов графа.
            По умолчанию используются все узлы графа

        `data_sample_size`: int = None
            Размер выборки данных для расчета. Количество записей
            data ,которые будут учтены при расчете.
        
        `data_node_field`: str   = 'node'
            Поле в котором хранится индекс узла, для которого производится расчет.

        `weight`: str или callable  = "travel_time"
            Имя поля содержащего вес ребер, или функция позволяющая вычислять 
            вес динамически.

        `cutoff`: float = None,
            Размер расчетной области. По умолчанию производится расчет для всего графа.

        `delay`: float, optional = None
            Задержка в расчете. Например на обслуживание вызова на пожар.
            По умолчанию указана в swiss_knife.DELAY_TIME

        `target_set`: set = None
            Целевой сет узлов графа, которые рассматриваются в качестве потенциальных мест размещения.

        `print_calc_states`: bool = True
            Флаг отображения прогресса расчета.
        
        `calc_function`: callable
            Функция запускаемая на каждой из добавленных подразделений

        `batch_size`: int
            Размер батча при расчете. Позволяет избежать проблемы переполнения памяти

    Возвращает:

        `matrix`: pd.DataFrame
            Время прибытия первого подразделения в каждый из узлов графа. 
            Столбцами являются узлы графа, индексами - узлы графа, до которых производился расчет.
    '''

    # Если конкретный набор данных не передан, используем узлы графа
    if data is None:
        data = ox.graph_to_gdfs(G, edges=False)
        data[data_node_field] = data.index

    # Если указана доля набора данных, выбираем его случайным образом из data
    if not data_sample_size is None:
        data = data.sample(data_sample_size)

    # Реверсный граф
    GR = nx.reverse(G, copy=False)

    # Матрица
    matrix = pd.DataFrame()

    # перебираем все записи в наборе данных
    if print_calc_states:
        pb = Progressbar(len(data), bins=40)
    i = 0
    for bi in range(0, len(data), batch_size):
        print('НАЧАЛО БАТЧА', bi, bi+batch_size)
        d = {}
        data_batch = data.iloc[bi:bi+batch_size]

        for di, dt in data_batch.iterrows():
            node = dt[data_node_field]
            if not data_cutoff_field is None:
                ctf = dt[data_cutoff_field]
            else:
                ctf = cutoff - delay if not cutoff is None else None
            length = nx.single_source_dijkstra_path_length(
                GR,
                source = node,
                cutoff = ctf,
                weight = weight,
                )
            # Оставляем только узлы в которых можно разместить [предполагалось удалить позже, но почему, пока не понятно]
            if not target_set is None:
                length = {k:v for k,v in length.items() if k in target_set}
                
            d[di] = pd.Series(length) + delay
            # d[node] = pd.Series(length) + delay
            if print_calc_states:
                pb()
            if calc_function:
                value = i / len(data)
                calc_function(value = value)

            i += 1

        dft = pd.DataFrame.from_dict(d, orient='index')
        if len(matrix) == 0:
            matrix = dft
        else:
            matrix = pd.concat([matrix, dft])
        print('КОНЕЦ БАТЧА', bi, bi+batch_size)

    # Формирование матрицы
    # matrix = pd.DataFrame.from_dict(d, orient='index')

    # Удаление зданий, к которым невозможно прибытия из перечня приемлемых узлов
    if not target_set is None:
        # Оставляем только узлы в которых можно разместить
        # matrix = matrix[list(set(matrix.columns) & target_set)]
        # Здания к которым невозможно прибыть после отброса
        zero_buildings = np.sum(matrix, axis=1)==0
        zero_buildings_count = sum(zero_buildings)
        if zero_buildings_count > 0:
            print(f'!{zero_buildings_count} зданий не доступны из указанных мест размещения!')
            matrix = matrix[zero_buildings == False]

    G = nx.reverse(GR, copy=False)
    return matrix

#### Тест расчета

In [12]:
%%time

# Начать отслеживание
tracemalloc.start()

# Тестируемый код
matrix = get_atm(
    G,
    data             = buildings,
    cutoff           = 10,
    data_sample_size = 10000
)

# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

# Печать итогового размера матрицы
print(matrix.shape)

НАЧАЛО БАТЧА 0 1000
КОНЕЦ БАТЧА 0 1000_______________________| 10.0%
НАЧАЛО БАТЧА 1000 2000
КОНЕЦ БАТЧА 1000 2000____________________| 20.0%
НАЧАЛО БАТЧА 2000 3000
КОНЕЦ БАТЧА 2000 3000____________________| 30.0%
НАЧАЛО БАТЧА 3000 4000
КОНЕЦ БАТЧА 3000 4000____________________| 40.0%
НАЧАЛО БАТЧА 4000 5000
КОНЕЦ БАТЧА 4000 5000____________________| 50.0%
НАЧАЛО БАТЧА 5000 6000
КОНЕЦ БАТЧА 5000 6000####________________| 60.0%
НАЧАЛО БАТЧА 6000 7000
КОНЕЦ БАТЧА 6000 7000########____________| 70.0%
НАЧАЛО БАТЧА 7000 8000
КОНЕЦ БАТЧА 7000 8000############________| 80.0%
НАЧАЛО БАТЧА 8000 9000
КОНЕЦ БАТЧА 8000 9000################____| 90.0%
НАЧАЛО БАТЧА 9000 10000
|########################################| 100.0%
КОНЕЦ БАТЧА 9000 10000
Текущее использование: 7784.74 MB
Пиковое использование: 14863.57 MB
(10000, 91695)
CPU times: total: 1h 25min 32s
Wall time: 1h 26min 30s


### Вариант 4 

Мультипроцессинг

#### Модификация get_atm()

In [9]:
from multiprocessing import Pool, cpu_count
import networkx as nx

from genesis.swiss_knife import DELAY_TIME
from genesis.utils import Progressbar

def process_batch(args):
    """Обрабатывает один батч данных"""
    GR, batch_data, batch_number, data_node_field, data_cutoff_field, cutoff, delay, weight, target_set = args
    
    print('БАТЧ ЗАПУЩЕН!', batch_number)

    batch_results = {}
    
    for row in batch_data.itertuples():
        node = getattr(row, data_node_field)
        
        # Определяем cutoff
        if data_cutoff_field is not None:
            ctf = getattr(row, data_cutoff_field)
        else:
            ctf = cutoff - delay if cutoff is not None else None
        
        # Расчет
        length = nx.single_source_dijkstra_path_length(
            GR,
            source=node,
            cutoff=ctf,
            weight=weight,
        )
        
        if target_set is not None:
            length = {k: v for k, v in length.items() if k in target_set}

        batch_results[row.Index] = {k: v + delay for k, v in length.items()}
    
    return batch_results

def get_atm_parallel(G,
                     data: gpd.GeoDataFrame = None,
                     data_sample_size: int = None,
                     data_node_field: str = 'node',
                     weight: str = 'travel_time',
                     cutoff: float = None,
                     data_cutoff_field: str = None,
                     delay: float = DELAY_TIME,
                     target_set: set = None,
                     batch_size: int = 1000,
                     n_jobs: int = None,  # Количество процессов
                     ):
    # Подготовка данных
    if data is None:
        data = ox.graph_to_gdfs(G, edges=False)
        data[data_node_field] = data.index

    if data_sample_size is not None:
        data = data.sample(data_sample_size)

    GR = nx.reverse(G, copy=False)
    
    # Создание батчей
    n_batches = len(data) // batch_size + (1 if len(data) % batch_size > 0 else 0)
    batches = [data.iloc[i*batch_size:(i+1)*batch_size] for i in range(n_batches)]
    
    # Подготовка аргументов
    args = [(GR, batch, batch_number, data_node_field, data_cutoff_field, cutoff, delay, weight, target_set) 
             for batch, batch_number in zip(batches, range(0,len(batches)))]
    
    # Параллельная обработка
    if n_jobs is None:
        n_jobs = cpu_count() - 1
    
    with Pool(n_jobs) as pool:
        batch_results_list = pool.map(process_batch, args)
    
    # Объединение результатов
    d = {}
    for batch_results in batch_results_list:
        d.update(batch_results)
    
    matrix = pd.DataFrame.from_dict(d, orient='index')
    
    # Фильтрация
    if target_set is not None:
        zero_buildings = np.sum(matrix, axis=1) == 0
        if sum(zero_buildings) > 0:
            matrix = matrix[zero_buildings == False]
    
    G = nx.reverse(GR, copy=False)

    return matrix

#### Тест расчета

In [ ]:
%%time

# Начать отслеживание
tracemalloc.start()

# Тестируемый код
matrix = get_atm_parallel(
    G,
    data             = buildings,
    cutoff           = 10,
    data_sample_size = 10000
)

# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

# Печать итогового размера матрицы
print(matrix.shape)